In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import nibabel as nib
import os
import seaborn as sns
from pathlib import Path

# Patch row_stack back into NumPy before importing brainsmash
if not hasattr(np, 'row_stack'):
    np.row_stack = np.vstack
    
from brainsmash.workbench.geo import volume
from brainsmash.mapgen.eval import sampled_fit
from brainsmash.mapgen.base import Base
from brainsmash.mapgen.memmap import txt2memmap
from brainsmash.mapgen.stats import nonparp

In [ ]:
#load the first map and desired masks
input_folder = Path("data/derivatives").resolve()
mp2rage_r1_path = input_folder / Path("3T/MP2RAGE/sub-2173019normasummer/ses-1/acq-mp2ragewipsag1isoc6cs7DL6820us_rec-DL/"
                                      "coreg/sub-2173019normasummer_ses-1_acq-mp2ragewipsag1isoc6cs7DL6820us_rec-DL_"
                                      "R1map_b1corr_coreg.nii.gz")

mp2rage_r1_img = nib.load(mp2rage_r1_path)
mp2rage_r1 = mp2rage_r1_img.get_fdata()
affine = mp2rage_r1_img.affine
valid = np.isfinite(mp2rage_r1) #mask to exclude infinity and NaN values

white_matter_path = input_folder / Path("3T/MP2RAGE/sub-2173019normasummer/ses-1/acq-mp2ragewipsag1isoc6cs7DL6820us_rec-DL/"
                                        "coreg/sub-2173019normasummer_ses-1_acq-mp2ragewipsag1isoc6cs7DL6820us_rec-DL_"
                                        "wm90percent_mask_reg2fs.nii.gz")
white_matter = nib.load(white_matter_path).get_fdata().astype(bool)
                                        
aparc_aseg_path = input_folder / Path("3T/freesurfer/sub-2173019normasummer_ses-1_acq-mp2ragewipsag1isoc6DL5340us_rec-DL/"
                                      "mri/aparc+aseg.nii.gz")                                      
aparc_aseg = nib.load(aparc_aseg_path).get_fdata().astype(np.uint16)
cc_anterior = np.isin(aparc_aseg, 255).astype(bool)
caudate = np.isin(aparc_aseg, [11,50]).astype(bool)

In [ ]:
out_folder = Path("data/derivatives/brainsmash").resolve()
mask_dict = {
    "white_matter": white_matter, 
    "cc_anterior": cc_anterior, 
    "caudate": caudate}

# Extract the physical 3D coordinates (X, Y, Z in mm) for voxels within each mask
for name, mask in mask_dict.items():
    indices = np.argwhere(mask)
    coordinates = [affine @ np.append(idx, 1) for idx in indices]
    valid_mask = valid[mask]
    coordinates = np.array(coordinates)[:, :3][valid_mask]
    coordinates_txt = out_folder / Path(name) / Path('coordinates.txt')
    os.makedirs(coordinates_txt.parent, exist_ok=True)
    np.savetxt(coordinates_txt, coordinates)

In [ ]:
#create list of map paths
qMT_r1_path = input_folder / Path("3T/qMT/sub-2173019normasummer/ses-1/acq-sag1isoc9/"
                                  "reg2MP2RAGE/sub-2173019normasummer_ses-1_acq-sag1isoc9_"
                                  "R1map_reg2FSmp2ragewipsag1isoc6DL5340us_rec-DL.nii.gz")
dki_fa_path = input_folder / Path("3T/dwi/sub-2173019normasummer/ses-1/acq-DWIepse2/"
                                  "reg2MP2RAGE/dki/sub-2173019normasummer_ses-1_acq-DWIepse2_"
                                  "fa_reg2FSmp2ragewipsag1isoc6DL5340us_rec-DL.nii.gz")
dki_kfa_path = input_folder / Path("3T/dwi/sub-2173019normasummer/ses-1/acq-DWIepse2/"
                                  "reg2MP2RAGE/dki/sub-2173019normasummer_ses-1_acq-DWIepse2_"
                                  "kfa_reg2FSmp2ragewipsag1isoc6DL5340us_rec-DL.nii.gz")
dki_md_path = input_folder / Path("3T/dwi/sub-2173019normasummer/ses-1/acq-DWIepse2/"
                                  "reg2MP2RAGE/dki/sub-2173019normasummer_ses-1_acq-DWIepse2_"
                                  "md_reg2FSmp2ragewipsag1isoc6DL5340us_rec-DL.nii.gz")
dki_mkt_path = input_folder / Path("3T/dwi/sub-2173019normasummer/ses-1/acq-DWIepse2/"
                                  "reg2MP2RAGE/dki/sub-2173019normasummer_ses-1_acq-DWIepse2_"
                                  "mkt_reg2FSmp2ragewipsag1isoc6DL5340us_rec-DL.nii.gz")
ihmtr_cosmod_path = input_folder / Path("3T/ihmt/sub-2173019normasummer/ses-1/acq-ihMTMCsag16isop3/"
                                        "reg2MP2RAGE/sub-2173019normasummer_ses-1_acq-ihMTMCsag16isop3_"
                                        "cosmod_ihMTR_reg2FSmp2ragewipsag1isoc6DL5340us_rec-DL.nii.gz")
ihmtr_freqalt_path = input_folder / Path("3T/ihmt/sub-2173019normasummer/ses-1/acq-ihMTMCsag16isop3/"
                                        "reg2MP2RAGE/sub-2173019normasummer_ses-1_acq-ihMTMCsag16isop3_"
                                        "freqalt_ihMTR_reg2FSmp2ragewipsag1isoc6DL5340us_rec-DL.nii.gz")
qsm_chimap_path = input_folder / Path("3T/QSM/derivatives/qsmxt/sub-2173019normasummer/ses-1/anat/"
                                     "reg2MP2RAGE/sub-2173019normasummer_ses-1_acq-sag1isoc9_"
                                     "MEGRE_Chimap_reg2FSmp2ragewipsag1isoc6DL5340us_rec-DL.nii.gz")
qsm_r2star_path = input_folder / Path("3T/QSM/derivatives/qsmxt/sub-2173019normasummer/ses-1/anat/"
                                     "reg2MP2RAGE/sub-2173019normasummer_ses-1_acq-sag1isoc9_"
                                     "MEGRE_R2starmap_reg2FSmp2ragewipsag1isoc6DL5340us_rec-DL.nii.gz")
qsm_swi_path = input_folder / Path("3T/QSM/derivatives/qsmxt/sub-2173019normasummer/ses-1/anat/"
                                     "reg2MP2RAGE/sub-2173019normasummer_ses-1_acq-sag1isoc9_"
                                     "MEGRE_swi_reg2FSmp2ragewipsag1isoc6DL5340us_rec-DL.nii.gz")
map_paths_dict = {
    "mp2rage_r1" : mp2rage_r1_path,
    "qMT_r1" : qMT_r1_path,
    "dki_fa" : dki_fa_path,
    "dki_kfa" : dki_kfa_path,
    "dki_md" : dki_md_path,
    "dki_mkt" : dki_mkt_path,
    "ihmtr_cosmod" : ihmtr_cosmod_path,
    "ihmtr_freqalt" : ihmtr_freqalt_path,
    "qsm_chimap" : qsm_chimap_path,
    "qsm_r2star" : qsm_r2star_path,
    "qsm_swi" : qsm_swi_path
}

In [ ]:
#mask maps and save as flat txt files
for map_name, map_path in map_paths_dict.items():
    img = nib.load(map_path)
    for mask_name, mask in mask_dict.items():
        valid_mask = valid[mask]
        img.get_fdata()[valid_mask]
        map_txt = out_folder / Path(mask_name) / Path(f"{map_name}.txt")

In [ ]:
valid_mask

In [ ]:
# Extract the physical 3D coordinates (X, Y, Z in mm) for all WM voxels
# Using the affine matrix


# 2. Mask the mp2rage r1 and clean up NaNs and Infinite values
mp2rage_r1_cc = mp2rage_r1[cc_anterior]
mp2rage_r1_caudate = mp2rage_r1[caudate]
valid_cc = np.isfinite(mp2rage_r1_cc)
valid_caudate = np.isfinite(mp2rage_r1_caudate)
coordinates_cc = coordinates_cc[valid_cc]
coordinates_caudate = coordinates_caudate[valid_caudate]
mp2rage_r1_cc = mp2rage_r1_cc[valid_cc]
mp2rage_r1_caudate = mp2rage_r1_caudate[valid_caudate]

# 3. Save to disk as txt files for brainsmash


mp2rage_r1_cc_txt = '/home/rflaherty/normabrain/data/derivatives/3T/brainsmash/cc_anterior/mp2rage_r1_cc.txt'
mp2rage_r1_caudate_txt = '/home/rflaherty/normabrain/data/derivatives/3T/brainsmash/caudate/mp2rage_r1_caudate.txt'
np.savetxt(mp2rage_r1_cc_txt, mp2rage_r1_cc)
np.savetxt(mp2rage_r1_caudate_txt, mp2rage_r1_caudate)

In [ ]:
# Use BrainSMASH's volume generator to build memory-mapped distance matrix files
filenames_cc = volume(coordinates_cc_txt, out_dir_cc)

In [ ]:
filenames_caudate = volume(coordinates_caudate_txt, out_dir_caudate)

In [ ]:
kwargs_cc = {'ns': 100, 'knn': 75, 'pv': 70} # Standard hyperparameters for dense volumes
sampled_fit(mp2rage_r1_cc_txt, filenames_cc['D'], filenames_cc['index'], nsurr=10, **kwargs_cc)

In [ ]:
kwargs_caudate = {'ns': 100, 'knn': 75, 'pv': 70} # Standard hyperparameters for dense volumes
sampled_fit(mp2rage_r1_caudate_txt, filenames_caudate['D'], filenames_caudate['index'], nsurr=10, **kwargs_caudate)

In [ ]:
# If the fit looks good, generate your 1,000 surrogate null maps!
gen_cc = Sampled(x=mp2rage_r1_cc_txt, D=filenames_cc['D'], index=filenames_cc['index'], **kwargs_cc)
surrogate_maps_cc = gen_cc(n=1000) 

In [ ]:
# If the fit looks good, generate your 1,000 surrogate null maps!
gen_caudate = Sampled(x=mp2rage_r1_caudate_txt, D=filenames_caudate['D'], index=filenames_caudate['index'], **kwargs_caudate)
surrogate_maps_caudate = gen_caudate(n=1000) 

In [ ]:
#load other maps, mask, and save as txt
qMT_r1_img = nib.load("/home/rflaherty/normabrain/data/derivatives/3T/qMT/sub-2173019normasummer/ses-1/acq-sag1isoc9/reg2MP2RAGE/sub-2173019normasummer_ses-1_acq-sag1isoc9_R1map_reg2FSmp2ragewipsag1isoc6DL5340us_rec-DL.nii.gz")
dki_fa_img = nib.load("/home/rflaherty/normabrain/data/derivatives/3T/dwi/sub-2173019normasummer/ses-1/acq-DWIepse2/reg2MP2RAGE/dki/sub-2173019normasummer_ses-1_acq-DWIepse2_fa_reg2FSmp2ragewipsag1isoc6DL5340us_rec-DL.nii.gz")
dki_kfa_img = nib.load("/home/rflaherty/normabrain/data/derivatives/3T/dwi/sub-2173019normasummer/ses-1/acq-DWIepse2/reg2MP2RAGE/dki/sub-2173019normasummer_ses-1_acq-DWIepse2_kfa_reg2FSmp2ragewipsag1isoc6DL5340us_rec-DL.nii.gz")
dki_md_img = nib.load("/home/rflaherty/normabrain/data/derivatives/3T/dwi/sub-2173019normasummer/ses-1/acq-DWIepse2/reg2MP2RAGE/dki/sub-2173019normasummer_ses-1_acq-DWIepse2_md_reg2FSmp2ragewipsag1isoc6DL5340us_rec-DL.nii.gz")
dki_mkt_img = nib.load("/home/rflaherty/normabrain/data/derivatives/3T/dwi/sub-2173019normasummer/ses-1/acq-DWIepse2/reg2MP2RAGE/dki/sub-2173019normasummer_ses-1_acq-DWIepse2_mkt_reg2FSmp2ragewipsag1isoc6DL5340us_rec-DL.nii.gz")
ihmtr_cosmod_img = nib.load("/home/rflaherty/normabrain/data/derivatives/3T/ihmt/sub-2173019normasummer/ses-1/acq-ihMTMCsag16isop3/reg2MP2RAGE/sub-2173019normasummer_ses-1_acq-ihMTMCsag16isop3_cosmod_ihMTR_reg2FSmp2ragewipsag1isoc6DL5340us_rec-DL.nii.gz")
ihmtr_freqalt_img = nib.load("/home/rflaherty/normabrain/data/derivatives/3T/ihmt/sub-2173019normasummer/ses-1/acq-ihMTMCsag16isop3/reg2MP2RAGE/sub-2173019normasummer_ses-1_acq-ihMTMCsag16isop3_freqalt_ihMTR_reg2FSmp2ragewipsag1isoc6DL5340us_rec-DL.nii.gz")
qsm_chimap_img = nib.load("/home/rflaherty/normabrain/data/derivatives/3T/QSM/derivatives/qsmxt/sub-2173019normasummer/ses-1/anat/reg2MP2RAGE/sub-2173019normasummer_ses-1_acq-sag1isoc9_MEGRE_Chimap_reg2FSmp2ragewipsag1isoc6DL5340us_rec-DL.nii.gz")
qsm_r2star_img = nib.load("/home/rflaherty/normabrain/data/derivatives/3T/QSM/derivatives/qsmxt/sub-2173019normasummer/ses-1/anat/reg2MP2RAGE/sub-2173019normasummer_ses-1_acq-sag1isoc9_MEGRE_R2starmap_reg2FSmp2ragewipsag1isoc6DL5340us_rec-DL.nii.gz")
qsm_swi_img = nib.load("/home/rflaherty/normabrain/data/derivatives/3T/QSM/derivatives/qsmxt/sub-2173019normasummer/ses-1/anat/reg2MP2RAGE/sub-2173019normasummer_ses-1_acq-sag1isoc9_MEGRE_swi_reg2FSmp2ragewipsag1isoc6DL5340us_rec-DL.nii.gz")

qMT_r1_cc = qMT_r1_img.get_fdata()[cc_anterior][valid_cc]
dki_fa_cc = dki_fa_img.get_fdata()[cc_anterior][valid_cc]
dki_kfa_cc = dki_kfa_img.get_fdata()[cc_anterior][valid_cc]
dki_md_cc = dki_md_img.get_fdata()[cc_anterior][valid_cc]
dki_mkt_cc = dki_mkt_img.get_fdata()[cc_anterior][valid_cc]
ihmtr_cosmod_cc = ihmtr_cosmod_img.get_fdata()[cc_anterior][valid_cc]
ihmtr_freqalt_cc = ihmtr_freqalt_img.get_fdata()[cc_anterior][valid_cc]
qsm_chimap_cc = qsm_chimap_img.get_fdata()[cc_anterior][valid_cc]
qsm_r2star_cc = qsm_r2star_img.get_fdata()[cc_anterior][valid_cc]
qsm_swi_cc = qsm_swi_img.get_fdata()[cc_anterior][valid_cc]

qMT_r1_cc_txt = '/home/rflaherty/normabrain/data/derivatives/3T/brainsmash/cc_anterior/qMT_r1_cc.txt'
dki_fa_cc_txt = '/home/rflaherty/normabrain/data/derivatives/3T/brainsmash/cc_anterior/dki_fa_cc.txt'
dki_kfa_cc_txt = '/home/rflaherty/normabrain/data/derivatives/3T/brainsmash/cc_anterior/dki_kfa_cc.txt'
dki_md_cc_txt = '/home/rflaherty/normabrain/data/derivatives/3T/brainsmash/cc_anterior/dki_md_cc.txt'
dki_mkt_cc_txt = '/home/rflaherty/normabrain/data/derivatives/3T/brainsmash/cc_anterior/dki_mkt_cc.txt'
ihmtr_cosmod_cc_txt = '/home/rflaherty/normabrain/data/derivatives/3T/brainsmash/cc_anterior/ihmtr_cosmod_cc.txt'
ihmtr_freqalt_cc_txt = '/home/rflaherty/normabrain/data/derivatives/3T/brainsmash/cc_anterior/ihmtr_freqalt_cc.txt'
qsm_chimap_cc_txt = '/home/rflaherty/normabrain/data/derivatives/3T/brainsmash/cc_anterior/qsm_chimap_cc.txt'
qsm_r2star_cc_txt = '/home/rflaherty/normabrain/data/derivatives/3T/brainsmash/cc_anterior/qsm_r2star_cc.txt'
qsm_swi_cc_txt = '/home/rflaherty/normabrain/data/derivatives/3T/brainsmash/cc_anterior/qsm_swi_cc.txt'

np.savetxt(qMT_r1_cc_txt, qMT_r1_cc)
np.savetxt(dki_fa_cc_txt, dki_fa_cc)
np.savetxt(dki_kfa_cc_txt, dki_kfa_cc)
np.savetxt(dki_md_cc_txt, dki_md_cc)
np.savetxt(dki_mkt_cc_txt, dki_mkt_cc)
np.savetxt(ihmtr_cosmod_cc_txt, ihmtr_cosmod_cc)
np.savetxt(ihmtr_freqalt_cc_txt, ihmtr_freqalt_cc)
np.savetxt(qsm_chimap_cc_txt, qsm_chimap_cc)
np.savetxt(qsm_r2star_cc_txt, qsm_r2star_cc)
np.savetxt(qsm_swi_cc_txt, qsm_swi_cc)

arr_mp2rage_r1_cc = np.loadtxt(mp2rage_r1_cc_txt)
arr_qMT_r1_cc = np.loadtxt(qMT_r1_cc_txt)
arr_dki_fa_cc = np.loadtxt(dki_fa_cc_txt)
arr_dki_kfa_cc = np.loadtxt(dki_kfa_cc_txt)
arr_dki_md_cc = np.loadtxt(dki_md_cc_txt)
arr_dki_mkt_cc = np.loadtxt(dki_mkt_cc_txt)
arr_ihmtr_cosmod_cc = np.loadtxt(ihmtr_cosmod_cc_txt)
arr_ihmtr_freqalt_cc = np.loadtxt(ihmtr_freqalt_cc_txt)
arr_qsm_chimap_cc = np.loadtxt(qsm_chimap_cc_txt)
arr_qsm_r2star_cc = np.loadtxt(qsm_r2star_cc_txt)
arr_qsm_swi_cc = np.loadtxt(qsm_swi_cc_txt)

maps_cc = np.vstack([arr_mp2rage_r1_cc,
                     arr_qMT_r1_cc, 
                              arr_dki_fa_cc, 
                              arr_dki_kfa_cc, 
                              arr_dki_md_cc,
                              arr_dki_mkt_cc,
                              arr_ihmtr_cosmod_cc,
                              arr_ihmtr_freqalt_cc,
                              arr_qsm_chimap_cc,
                              arr_qsm_r2star_cc,
                              arr_qsm_swi_cc
                             ])

n_maps_cc, n_voxels_cc = maps_cc.shape

In [ ]:
qMT_r1_caudate = qMT_r1_img.get_fdata()[caudate][valid_caudate]
dki_fa_caudate = dki_fa_img.get_fdata()[caudate][valid_caudate]
dki_kfa_caudate = dki_kfa_img.get_fdata()[caudate][valid_caudate]
dki_md_caudate = dki_md_img.get_fdata()[caudate][valid_caudate]
dki_mkt_caudate = dki_mkt_img.get_fdata()[caudate][valid_caudate]
ihmtr_cosmod_caudate = ihmtr_cosmod_img.get_fdata()[caudate][valid_caudate]
ihmtr_freqalt_caudate = ihmtr_freqalt_img.get_fdata()[caudate][valid_caudate]
qsm_chimap_caudate = qsm_chimap_img.get_fdata()[caudate][valid_caudate]
qsm_r2star_caudate = qsm_r2star_img.get_fdata()[caudate][valid_caudate]
qsm_swi_caudate = qsm_swi_img.get_fdata()[caudate][valid_caudate]

qMT_r1_caudate_txt = '/home/rflaherty/normabrain/data/derivatives/3T/brainsmash/caudate/qMT_r1_caudate.txt'
dki_fa_caudate_txt = '/home/rflaherty/normabrain/data/derivatives/3T/brainsmash/caudate/dki_fa_caudate.txt'
dki_kfa_caudate_txt = '/home/rflaherty/normabrain/data/derivatives/3T/brainsmash/caudate/dki_kfa_caudate.txt'
dki_md_caudate_txt = '/home/rflaherty/normabrain/data/derivatives/3T/brainsmash/caudate/dki_md_caudate.txt'
dki_mkt_caudate_txt = '/home/rflaherty/normabrain/data/derivatives/3T/brainsmash/caudate/dki_mkt_caudate.txt'
ihmtr_cosmod_caudate_txt = '/home/rflaherty/normabrain/data/derivatives/3T/brainsmash/caudate/ihmtr_cosmod_caudate.txt'
ihmtr_freqalt_caudate_txt = '/home/rflaherty/normabrain/data/derivatives/3T/brainsmash/caudate/ihmtr_freqalt_caudate.txt'
qsm_chimap_caudate_txt = '/home/rflaherty/normabrain/data/derivatives/3T/brainsmash/caudate/qsm_chimap_caudate.txt'
qsm_r2star_caudate_txt = '/home/rflaherty/normabrain/data/derivatives/3T/brainsmash/caudate/qsm_r2star_caudate.txt'
qsm_swi_caudate_txt = '/home/rflaherty/normabrain/data/derivatives/3T/brainsmash/caudate/qsm_swi_caudate.txt'

np.savetxt(qMT_r1_caudate_txt, qMT_r1_caudate)
np.savetxt(dki_fa_caudate_txt, dki_fa_caudate)
np.savetxt(dki_kfa_caudate_txt, dki_kfa_caudate)
np.savetxt(dki_md_caudate_txt, dki_md_caudate)
np.savetxt(dki_mkt_caudate_txt, dki_mkt_caudate)
np.savetxt(ihmtr_cosmod_caudate_txt, ihmtr_cosmod_caudate)
np.savetxt(ihmtr_freqalt_caudate_txt, ihmtr_freqalt_caudate)
np.savetxt(qsm_chimap_caudate_txt, qsm_chimap_caudate)
np.savetxt(qsm_r2star_caudate_txt, qsm_r2star_caudate)
np.savetxt(qsm_swi_caudate_txt, qsm_swi_caudate)

arr_mp2rage_r1_caudate = np.loadtxt(mp2rage_r1_caudate_txt)
arr_qMT_r1_caudate = np.loadtxt(qMT_r1_caudate_txt)
arr_dki_fa_caudate = np.loadtxt(dki_fa_caudate_txt)
arr_dki_kfa_caudate = np.loadtxt(dki_kfa_caudate_txt)
arr_dki_md_caudate = np.loadtxt(dki_md_caudate_txt)
arr_dki_mkt_caudate = np.loadtxt(dki_mkt_caudate_txt)
arr_ihmtr_cosmod_caudate = np.loadtxt(ihmtr_cosmod_caudate_txt)
arr_ihmtr_freqalt_caudate = np.loadtxt(ihmtr_freqalt_caudate_txt)
arr_qsm_chimap_caudate = np.loadtxt(qsm_chimap_caudate_txt)
arr_qsm_r2star_caudate = np.loadtxt(qsm_r2star_caudate_txt)
arr_qsm_swi_caudate = np.loadtxt(qsm_swi_caudate_txt)

maps_caudate = np.vstack([arr_mp2rage_r1_caudate,
                                      arr_qMT_r1_caudate, 
                              arr_dki_fa_caudate, 
                              arr_dki_kfa_caudate, 
                              arr_dki_md_caudate,
                              arr_dki_mkt_caudate,
                              arr_ihmtr_cosmod_caudate,
                              arr_ihmtr_freqalt_caudate,
                              arr_qsm_chimap_caudate,
                              arr_qsm_r2star_caudate,
                              arr_qsm_swi_caudate
                             ])

n_maps_caudate, n_voxels_caudate = maps_caudate.shape

In [ ]:
dataset_names = ["MP2RAGE R1", 'qMT R1', 
                 'DKI FA', 'DKI KFA', 'DKI MD', 'DKI MKT',
                 'ihMTR CosMod', 'ihMTR Freq Alt',
                 'QSM Chi', 'R2*', 'SWI']

In [ ]:
#initialize matrices
observed_matrix_cc = np.zeros((n_maps, n_maps))
p_value_matrix_cc = np.zeros((n_maps, n_maps))
annot_labels_cc = np.empty((n_maps, n_maps), dtype=object)

kwargs_caudate = {'ns': 100, 'knn': 75, 'pv': 70} # Standard hyperparameters for dense volumes
#compute pairwise correlations and corrected p values
for i in range(n_maps_cc):
    gen = Sampled(x=maps_cc[i], D=filenames_cc['D'], index=filenames_cc['index'], **kwargs_cc)
    surrogates_i = gen(n=500)

    row_scaled = (maps_cc[i] - maps_cc[i].mean()) / maps_cc[i].std()
    surr_scaled = (surrogates_i - surrogates_i.mean(axis=1, keepdims=True)) / surrogates_i.std(axis=1, keepdims=True)

    for j in range(n_maps):
        if i == j:
            #diagonal is perfectly correlated
            observed_matrix_cc[i, j] = 1.0
            p_value_matrix_cc[i, j] = 0.0
            annot_labels_cc[i, j] = "r = 1.00\n--"
            continue
        col_scaled = (maps_cc[j] - maps_cc[j].mean()) / maps_cc[j].std()
        r_obs = np.dot(row_scaled, col_scaled) / n_voxels_cc
        observed_matrix_cc[i, j] = r_obs
        null_dist = np.dot(surr_scaled, col_scaled) / n_voxels_cc
        p_val = nonparp(r_obs, null_dist)
        p_value_matrix_cc[i, j] = p_val

        # Format strings for the heatmap text overlay
        stars = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else " (ns)"
        p_str = f"p < .001" if p_val < 0.001 else f"p = {p_val:.3f}"
        annot_labels_cc[i, j] = f"r = {r_obs:.2f}\n{p_str}\n{stars}"

In [ ]:
#Plot the Volumetric Pairwise Heatmap
plt.figure(figsize=(12, 13))
sns.heatmap(
    observed_matrix_cc,
    mask=np.triu(np.ones_like(observed_matrix_cc, dtype=bool), k=1),
    annot=annot_labels_cc,
    fmt="",
    cmap="RdBu_r",
    vmin=-1.0,
    vmax=1.0,
    center=0,
    xticklabels=dataset_names,
    yticklabels=dataset_names,
    cbar_kws={"label": "Volumetric Spatial Correlation (r)"},
    linewidths=1,
    linecolor="black",
)

plt.title("Anterior Corpus Callosum Pairwise Correlation Matrix\n(Spatial Autocorrelation Corrected)", fontsize=12, pad=20)
plt.tight_layout()
plt.savefig("anterior_cc_corr.pdf", bbox_inches="tight")
plt.show()